In [33]:
%pip install ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [34]:
import pandas as pd
import re

# Load movies.csv
movies = pd.read_csv("movies.csv")

# Clean movie titles (lowercase, remove punctuation)
def clean_title(title):
    title = title.lower()
    return re.sub(r"[^a-z0-9 ]", "", title)

movies["clean_title"] = movies["title"].apply(clean_title)

# Combine title + genres into a single field for vectorizing
movies["combined"] = movies["title"] + " " + movies["genres"].fillna("")

In [35]:
from sklearn.feature_extraction.text import CountVectorizer

# Create a CountVectorizer model
vectorizer = CountVectorizer(ngram_range=(1, 2), stop_words="english")

# Fit and transform the combined title+genre column
combined_matrix = vectorizer.fit_transform(movies["combined"])

In [36]:
# Load ratings.csv
ratings = pd.read_csv("ratings.csv")

# Preview (optional)
ratings.head()

,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828
3,1,665,5.0,1147878820
4,1,899,3.5,1147868510


In [37]:
def get_collaborative_scores(movie_ids, min_rating=4.0, min_overlap=0.1):
    # Step 1: Find users who rated any of the given movies highly
    liked_users = ratings[
        (ratings["movieId"].isin(movie_ids)) & (ratings["rating"] >= min_rating)
    ]["userId"].unique()

    if len(liked_users) == 0:
        return pd.Series()

    # Step 2: Get other movies these users also rated highly
    similar_ratings = ratings[
        (ratings["userId"].isin(liked_users)) & (ratings["rating"] >= min_rating)
    ]

    # Step 3: Count how often each movie was liked
    movie_like_freq = similar_ratings["movieId"].value_counts()

    # Normalize by number of users to get score
    collaborative_score = movie_like_freq / len(liked_users)

    # Optional: Filter low-overlap movies
    collaborative_score = collaborative_score[collaborative_score >= min_overlap]

    return collaborative_score

In [38]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def hybrid_recommendation(user_input, top_n=10):
    # Clean user input
    user_input = clean_title(user_input)

    # Step 1: Find all matching movie titles (partial match)
    matches = movies[movies["clean_title"].str.contains(user_input)]
    if matches.empty:
        return "No matching movies found."

    # Step 2: Get combined vector for all matches
    match_indices = matches.index.tolist()
    match_movie_ids = matches["movieId"].tolist()
    match_vectors = combined_matrix[match_indices]
    mean_vec = np.asarray(match_vectors.mean(axis=0))

    # Restrict cosine similarity only to title-matching candidates
    candidate_indices = movies[movies["clean_title"].str.contains(user_input)].index
    candidate_matrix = combined_matrix[candidate_indices]

    # Compute cosine similarity only with these candidates
    content_similarities = cosine_similarity(mean_vec, candidate_matrix).flatten()

    # Assign back only to matched candidates
    movies_copy = movies.iloc[candidate_indices].copy()
    movies_copy["content_score"] = content_similarities

    # Continue with collaborative scores
    match_movie_ids = movies_copy["movieId"].tolist()
    collab_scores = get_collaborative_scores(match_movie_ids)
    movies_copy["collab_score"] = movies_copy["movieId"].map(collab_scores).fillna(0)

    movies_copy["hybrid_score"] = (movies_copy["content_score"] * 0.7) + (movies_copy["collab_score"] * 0.3)
    
    # Step 4: Get collaborative scores
    collab_scores = get_collaborative_scores(match_movie_ids)

    # Step 5: Combine scores into DataFrame
    movies_copy = movies.copy()
    movies_copy["content_score"] = content_similarities

    # Add collaborative score (default 0 if missing)
    movies_copy["collab_score"] = movies_copy["movieId"].map(collab_scores).fillna(0)

    # Final hybrid score: weighted combination (you can tune weights)
    movies_copy["hybrid_score"] = (movies_copy["content_score"] * 0.7) + (movies_copy["collab_score"] * 0.3)

    # Step 6: Limit to movies that contain part of input in title (cleaned)
    filtered_recs = movies_copy[
        movies_copy["clean_title"].str.contains(user_input) & 
        (~movies_copy["movieId"].isin(match_movie_ids))
    ]

    # Step 7: If no filtered matches found, fallback to all others
    if filtered_recs.empty:
        filtered_recs = movies_copy[~movies_copy["movieId"].isin(match_movie_ids)]

    # Step 8: Sort and get top N
    final_recs = filtered_recs.sort_values("hybrid_score", ascending=False).head(top_n)

    return final_recs[["title", "genres", "content_score", "collab_score", "hybrid_score"]]